# MCP + OpenAI LLM + Resources

Este notebook extiende la demo anterior para mostrar cómo un LLM puede **usar resources MCP**.

La idea importante:

> En MCP, un **resource** no es una tool.  
> Un resource es contexto legible.  
> Si usamos un LLM con function calling, necesitamos decidir cómo le damos acceso a esos resources.

En esta demo usaremos una estrategia didáctica:

```text
LLM de OpenAI
  → pide leer un resource usando una function tool puente
  → Python llama a FastMCP Client.read_resource(...)
  → el contenido vuelve al LLM
  → el LLM responde usando ese contexto
```

Además de tools, agregaremos estos resources:

- `reglamento://aprobacion`
- `rubrica://trabajos/{trabajo}`
- `programa://materias/{materia_slug}`
- `calendario://examenes/2026`


## 1. Instalación


In [ ]:
%pip install -U fastmcp openai python-dotenv


## 2. Configurar OpenAI


In [ ]:
import os
import json
from getpass import getpass
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI
from fastmcp import FastMCP, Client

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Pegá tu OPENAI_API_KEY: ")

oai_client = OpenAI()

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.5")

MODEL


## 3. Datos simulados

Creamos una mini universidad ficticia.


In [ ]:
ALUMNOS: dict[str, dict[str, Any]] = {
    "A001": {
        "nombre": "Ana Torres",
        "carrera": "Ingeniería Informática",
        "email": "ana.torres@example.edu",
        "notas": [8, 9, 7],
        "entregas": {"TP1": True, "TP2": False, "TP3": True},
    },
    "A002": {
        "nombre": "Bruno López",
        "carrera": "Ingeniería Informática",
        "email": "bruno.lopez@example.edu",
        "notas": [6, 5, 7],
        "entregas": {"TP1": True, "TP2": True, "TP3": False},
    },
    "A003": {
        "nombre": "Carla Méndez",
        "carrera": "Ingeniería en Inteligencia Artificial",
        "email": "carla.mendez@example.edu",
        "notas": [10, 9, 10],
        "entregas": {"TP1": True, "TP2": True, "TP3": True},
    },
    "A004": {
        "nombre": "Diego Ruiz",
        "carrera": "Ingeniería Informática",
        "email": "diego.ruiz@example.edu",
        "notas": [3, 4, 5],
        "entregas": {"TP1": False, "TP2": False, "TP3": False},
    },
}

PROGRAMA_IA_GENERATIVA = {
    "materia": "IA Generativa",
    "codigo": "IA-402",
    "objetivos": [
        "Entender cómo se construyen aplicaciones con modelos de lenguaje.",
        "Diseñar herramientas, prompts y flujos de evaluación.",
        "Construir prototipos con APIs, RAG y MCP.",
    ],
    "unidades": [
        "Fundamentos de LLMs",
        "Prompting y evaluación",
        "RAG",
        "Agentes",
        "Model Context Protocol",
    ],
}

CALENDARIO_EXAMENES_2026 = {
    "parcial_1": "2026-05-12",
    "parcial_2": "2026-06-23",
    "recuperatorio": "2026-07-07",
    "entrega_final": "2026-07-14",
}

REGLAMENTO_APROBACION = """
# Reglamento simplificado de aprobación

Este documento es un recurso de lectura para la demo MCP.

## Condiciones

- La materia se aprueba con promedio final mayor o igual a 4.
- El estudiante debe tener al menos el 75% de asistencia.
- El estudiante debe entregar al menos 2 de los 3 trabajos prácticos.
- Si el promedio es menor a 4, el estudiante debe recuperar.
- Si adeuda más de un trabajo práctico, el equipo docente debe enviar un recordatorio personalizado.

## Reglas de comunicación

- Los recordatorios deben ser claros, respetuosos y accionables.
- No se deben enviar mails automáticos sin revisión humana.
- El asistente puede preparar borradores, pero una persona debe aprobar el envío.
""".strip()

RUBRICAS = {
    "TP1": {
        "criterios": [
            {"criterio": "claridad", "peso": 0.25},
            {"criterio": "corrección técnica", "peso": 0.35},
            {"criterio": "justificación", "peso": 0.20},
            {"criterio": "presentación", "peso": 0.20},
        ],
        "escala": "0 a 10",
    },
    "TP2": {
        "criterios": [
            {"criterio": "diseño de tools", "peso": 0.30},
            {"criterio": "validaciones", "peso": 0.25},
            {"criterio": "uso de resources", "peso": 0.25},
            {"criterio": "criterio de seguridad", "peso": 0.20},
        ],
        "escala": "0 a 10",
    },
    "TP3": {
        "criterios": [
            {"criterio": "integración con LLM", "peso": 0.30},
            {"criterio": "manejo de errores", "peso": 0.25},
            {"criterio": "experiencia de usuario", "peso": 0.25},
            {"criterio": "documentación", "peso": 0.20},
        ],
        "escala": "0 a 10",
    },
}


## 4. Crear MCP Server


In [ ]:
mcp = FastMCP(
    name="universidad-mcp",
    instructions=(
        "Servidor MCP educativo para una clase introductoria. "
        "Usá tools para acciones y resources para contexto de lectura. "
        "Las acciones de alto impacto deben producir borradores y requerir aprobación humana."
    ),
)


## 5. Agregar Tools

Las tools siguen siendo acciones. Las vamos a usar para consultar alumnos, calcular promedios y preparar borradores.


In [ ]:
@mcp.tool
def buscar_alumno(legajo: str) -> dict[str, Any]:
    """
    Busca un alumno por legajo.

    Args:
        legajo: Identificador del alumno. Ejemplo: "A001".

    Returns:
        Datos públicos del alumno para la demo, incluyendo notas y entregas.
    """
    alumno = ALUMNOS.get(legajo)

    if alumno is None:
        return {
            "encontrado": False,
            "legajo": legajo,
            "mensaje": "No existe un alumno con ese legajo.",
        }

    return {
        "encontrado": True,
        "legajo": legajo,
        "nombre": alumno["nombre"],
        "carrera": alumno["carrera"],
        "email": alumno["email"],
        "notas": alumno["notas"],
        "entregas": alumno["entregas"],
    }


@mcp.tool
def calcular_promedio(notas: list[float]) -> dict[str, Any]:
    """
    Calcula el promedio de una lista de notas entre 0 y 10.

    Args:
        notas: Lista de notas numéricas. Ejemplo: [8, 7.5, 9].

    Returns:
        Promedio, cantidad de notas y estado de aprobación.
    """
    if not notas:
        raise ValueError("La lista de notas no puede estar vacía.")

    for nota in notas:
        if nota < 0 or nota > 10:
            raise ValueError("Todas las notas deben estar entre 0 y 10.")

    promedio = round(sum(notas) / len(notas), 2)

    return {
        "promedio": promedio,
        "cantidad_notas": len(notas),
        "estado": "aprobado" if promedio >= 4 else "desaprobado",
    }


@mcp.tool
def buscar_entregas_pendientes(materia: str, trabajo: str) -> dict[str, Any]:
    """
    Busca estudiantes que no entregaron un trabajo práctico.

    Args:
        materia: Nombre de la materia. En esta demo usar "IA Generativa".
        trabajo: Nombre del trabajo. Ejemplo: "TP2".

    Returns:
        Lista de estudiantes con entrega pendiente.
    """
    if materia.lower() != "ia generativa":
        raise ValueError("Esta demo solo tiene datos de la materia 'IA Generativa'.")

    pendientes = []

    for legajo, alumno in ALUMNOS.items():
        entrego = alumno["entregas"].get(trabajo)

        if entrego is False:
            pendientes.append(
                {
                    "legajo": legajo,
                    "nombre": alumno["nombre"],
                    "email": alumno["email"],
                }
            )

    return {
        "materia": materia,
        "trabajo": trabajo,
        "cantidad_pendientes": len(pendientes),
        "pendientes": pendientes,
    }


@mcp.tool
def crear_borrador_recordatorio_entrega(
    materia: str,
    trabajo: str,
    destinatarios: list[str],
    mensaje_base: str = "Les recordamos que tienen una entrega pendiente.",
) -> dict[str, Any]:
    """
    Crea un borrador de recordatorio de entrega. No envía emails.

    Args:
        materia: Nombre de la materia.
        trabajo: Nombre del trabajo pendiente.
        destinatarios: Lista de emails destinatarios.
        mensaje_base: Mensaje inicial para incluir en el borrador.

    Returns:
        Borrador listo para revisión humana.
    """
    if not destinatarios:
        raise ValueError("Debe haber al menos un destinatario.")

    asunto = f"Recordatorio de entrega pendiente: {trabajo} - {materia}"

    cuerpo = f"""Hola,

{mensaje_base}

Materia: {materia}
Trabajo: {trabajo}

Por favor, revisen la consigna y avísennos si tienen alguna dificultad.

Saludos,
Equipo docente
"""

    return {
        "tipo": "borrador",
        "requiere_aprobacion_humana": True,
        "asunto": asunto,
        "destinatarios": destinatarios,
        "cuerpo": cuerpo,
        "advertencia": "Este borrador NO fue enviado. Debe revisarlo una persona.",
    }


## 6. Agregar Resources

Acá está el cambio importante.

Agregamos resources que representan **contexto legible**:

- programa de la materia
- calendario de exámenes
- reglamento de aprobación
- rúbrica dinámica por trabajo práctico


In [ ]:
@mcp.resource(
    "programa://materias/{materia_slug}",
    mime_type="application/json",
    annotations={"readOnlyHint": True, "idempotentHint": True},
)
def programa_materia(materia_slug: str) -> str:
    """Devuelve el programa de una materia."""
    if materia_slug != "ia-generativa":
        raise ValueError("Materia no disponible en esta demo.")

    return json.dumps(PROGRAMA_IA_GENERATIVA, ensure_ascii=False, indent=2)


@mcp.resource(
    "calendario://examenes/2026",
    mime_type="application/json",
    annotations={"readOnlyHint": True, "idempotentHint": True},
)
def calendario_examenes_2026() -> str:
    """Devuelve el calendario de exámenes 2026."""
    return json.dumps(CALENDARIO_EXAMENES_2026, ensure_ascii=False, indent=2)


@mcp.resource(
    "reglamento://aprobacion",
    mime_type="text/markdown",
    annotations={"readOnlyHint": True, "idempotentHint": True},
)
def reglamento_aprobacion() -> str:
    """Devuelve el reglamento simplificado de aprobación de la materia."""
    return REGLAMENTO_APROBACION


@mcp.resource(
    "rubrica://trabajos/{trabajo}",
    mime_type="application/json",
    annotations={"readOnlyHint": True, "idempotentHint": True},
)
def rubrica_trabajo(trabajo: str) -> str:
    """
    Devuelve la rúbrica de evaluación de un trabajo práctico.

    Args:
        trabajo: Nombre del trabajo. Ejemplo: "TP2".
    """
    rubrica = RUBRICAS.get(trabajo.upper())

    if rubrica is None:
        raise ValueError(f"No hay rúbrica disponible para {trabajo}.")

    return json.dumps(
        {
            "trabajo": trabajo.upper(),
            **rubrica,
        },
        ensure_ascii=False,
        indent=2,
    )


## 7. Agregar Prompt

El prompt es una plantilla reutilizable.


In [ ]:
@mcp.prompt
def feedback_entrega_tp(
    nombre_estudiante: str,
    trabajo: str,
    observaciones: str,
    tono: str = "cercano, claro y riguroso",
) -> str:
    """Genera una plantilla de feedback para una entrega práctica."""
    return f"""
Redactá feedback para {nombre_estudiante} sobre la entrega {trabajo}.

Tono: {tono}

Observaciones:
{observaciones}

Estructura obligatoria:
1. Empezá reconociendo algo positivo.
2. Señalá los problemas técnicos principales.
3. Explicá por qué esos problemas importan.
4. Proponé próximos pasos concretos.
5. Cerrá con una frase motivadora.
""".strip()


## 8. Probar lectura directa de Resources

Antes de involucrar al LLM, probamos que los resources funcionan.


In [ ]:
async with Client(mcp) as mcp_client:
    reglamento = await mcp_client.read_resource("reglamento://aprobacion")
    rubrica_tp2 = await mcp_client.read_resource("rubrica://trabajos/TP2")

print("REGLAMENTO:")
print(reglamento[0].text)

print("\nRÚBRICA TP2:")
print(rubrica_tp2[0].text)


## 9. Convertir Tools MCP a Tools de OpenAI

Igual que antes, convertimos las tools MCP normales al formato de OpenAI.


In [ ]:
def dump_model(obj: object) -> dict[str, Any]:
    if hasattr(obj, "model_dump"):
        return obj.model_dump(mode="json")
    if hasattr(obj, "dict"):
        return obj.dict()
    return obj.__dict__


def mcp_tool_to_openai_tool(tool: object) -> dict[str, Any]:
    data = dump_model(tool)

    name = data.get("name") or getattr(tool, "name")
    description = (
        data.get("description")
        or data.get("annotations", {}).get("title")
        or f"Ejecuta la herramienta MCP {name}."
    )

    parameters = (
        data.get("inputSchema")
        or data.get("input_schema")
        or data.get("parameters")
        or {
            "type": "object",
            "properties": {},
            "additionalProperties": False,
        }
    )

    return {
        "type": "function",
        "name": name,
        "description": description,
        "parameters": parameters,
    }


async with Client(mcp) as mcp_client:
    mcp_tools = await mcp_client.list_tools()

openai_tools = [mcp_tool_to_openai_tool(tool) for tool in mcp_tools]

print(json.dumps(openai_tools, ensure_ascii=False, indent=2))


## 10. Agregar Tools puente para Resources

Acá está la parte clave.

Como OpenAI function calling espera **functions**, vamos a crear dos function tools que no son tools de negocio, sino herramientas puente:

1. `listar_resources_mcp`
2. `leer_resource_mcp`

Estas funciones le permiten al LLM descubrir y leer resources MCP.


In [ ]:
resource_bridge_tools = [
    {
        "type": "function",
        "name": "listar_resources_mcp",
        "description": (
            "Lista los resources y resource templates disponibles en el servidor MCP. "
            "Usalo cuando necesites saber qué contexto de lectura existe."
        ),
        "parameters": {
            "type": "object",
            "properties": {},
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "leer_resource_mcp",
        "description": (
            "Lee un resource MCP por URI. Usalo para obtener contexto como reglamentos, "
            "programas, calendarios o rúbricas. Ejemplos: "
            "reglamento://aprobacion, rubrica://trabajos/TP2, "
            "programa://materias/ia-generativa, calendario://examenes/2026."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "uri": {
                    "type": "string",
                    "description": "URI exacta del resource MCP que se quiere leer.",
                }
            },
            "required": ["uri"],
            "additionalProperties": False,
        },
    },
]

openai_tools_with_resources = openai_tools + resource_bridge_tools

print(json.dumps(resource_bridge_tools, ensure_ascii=False, indent=2))


## 11. Router para Tools y Resources

Ahora el router puede hacer dos cosas:

- Si el modelo llama una tool MCP real, usa `call_tool`.
- Si el modelo llama una tool puente de resources, usa `list_resources` o `read_resource`.


In [ ]:
def resource_item_to_dict(item: object) -> dict[str, Any]:
    return {
        "uri": str(getattr(item, "uri", "")),
        "name": str(getattr(item, "name", "")),
        "description": str(getattr(item, "description", "")),
        "mime_type": str(getattr(item, "mimeType", getattr(item, "mime_type", ""))),
    }


def resource_content_to_dict(item: object) -> dict[str, Any]:
    text = getattr(item, "text", None)
    if text is None:
        text = getattr(item, "content", None)

    return {
        "uri": str(getattr(item, "uri", "")),
        "mime_type": str(getattr(item, "mimeType", getattr(item, "mime_type", ""))),
        "text": text,
    }


async def listar_resources_mcp() -> str:
    try:
        async with Client(mcp) as mcp_client:
            resources = await mcp_client.list_resources()

            templates = []
            if hasattr(mcp_client, "list_resource_templates"):
                try:
                    templates = await mcp_client.list_resource_templates()
                except Exception:
                    templates = []

        return json.dumps(
            {
                "ok": True,
                "resources": [resource_item_to_dict(r) for r in resources],
                "resource_templates": [resource_item_to_dict(t) for t in templates],
                "uris_de_ejemplo": [
                    "reglamento://aprobacion",
                    "rubrica://trabajos/TP2",
                    "programa://materias/ia-generativa",
                    "calendario://examenes/2026",
                ],
            },
            ensure_ascii=False,
        )

    except Exception as exc:
        return json.dumps(
            {
                "ok": False,
                "error": type(exc).__name__,
                "message": str(exc),
            },
            ensure_ascii=False,
        )


async def leer_resource_mcp(uri: str) -> str:
    try:
        async with Client(mcp) as mcp_client:
            content = await mcp_client.read_resource(uri)

        return json.dumps(
            {
                "ok": True,
                "uri": uri,
                "content": [resource_content_to_dict(item) for item in content],
            },
            ensure_ascii=False,
        )

    except Exception as exc:
        return json.dumps(
            {
                "ok": False,
                "uri": uri,
                "error": type(exc).__name__,
                "message": str(exc),
            },
            ensure_ascii=False,
        )


async def call_mcp_tool(tool_name: str, arguments: dict[str, Any]) -> str:
    try:
        async with Client(mcp) as mcp_client:
            result = await mcp_client.call_tool(tool_name, arguments)

        return json.dumps(
            {
                "ok": True,
                "tool": tool_name,
                "result": result.data,
            },
            ensure_ascii=False,
        )

    except Exception as exc:
        return json.dumps(
            {
                "ok": False,
                "tool": tool_name,
                "error": type(exc).__name__,
                "message": str(exc),
            },
            ensure_ascii=False,
        )


async def execute_openai_tool(tool_name: str, arguments: dict[str, Any]) -> str:
    if tool_name == "listar_resources_mcp":
        return await listar_resources_mcp()

    if tool_name == "leer_resource_mcp":
        return await leer_resource_mcp(arguments["uri"])

    return await call_mcp_tool(tool_name, arguments)


## 12. Agent loop con Tools + Resources

Este loop deja que el LLM use tanto tools de negocio como resources de contexto.


In [ ]:
SYSTEM_INSTRUCTIONS = """
Sos un asistente docente conectado a un MCP Server universitario.

Reglas:
- Usá tools cuando necesites ejecutar acciones o consultar datos dinámicos de alumnos, notas o entregas.
- Usá resources cuando necesites contexto estable: reglamentos, programa, calendario o rúbricas.
- No inventes legajos, emails, notas, entregas, fechas ni criterios de evaluación.
- Si necesitás saber qué resources existen, llamá listar_resources_mcp.
- Si necesitás leer contexto, llamá leer_resource_mcp con la URI adecuada.
- Si una acción puede afectar a estudiantes, prepará borradores y pedí aprobación humana.
- Nunca afirmes que un email fue enviado; la tool disponible solo crea borradores.
- Respondé en español, de forma clara y didáctica.
""".strip()


async def run_llm_with_mcp_and_resources(user_message: str, max_turns: int = 6) -> str:
    input_messages: list[Any] = [
        {
            "role": "user",
            "content": user_message,
        }
    ]

    for turn in range(max_turns):
        response = oai_client.responses.create(
            model=MODEL,
            instructions=SYSTEM_INSTRUCTIONS,
            input=input_messages,
            tools=openai_tools_with_resources,
            tool_choice="auto",
        )

        input_messages += response.output

        function_calls = [item for item in response.output if item.type == "function_call"]

        if not function_calls:
            return response.output_text

        print(f"\n--- Turno {turn + 1}: el modelo pidió {len(function_calls)} tool call(s) ---")

        for tool_call in function_calls:
            tool_name = tool_call.name
            args = json.loads(tool_call.arguments or "{}")

            print(f"🛠️ Tool call: {tool_name}")
            print(json.dumps(args, ensure_ascii=False, indent=2))

            tool_result = await execute_openai_tool(tool_name, args)

            print("📦 Resultado:")
            print(json.dumps(json.loads(tool_result), ensure_ascii=False, indent=2)[:2000])

            input_messages.append(
                {
                    "type": "function_call_output",
                    "call_id": tool_call.call_id,
                    "output": tool_result,
                }
            )

    raise RuntimeError("El loop alcanzó max_turns sin una respuesta final.")


## 13. Ejemplo 1: usar un resource de reglamento

El modelo debería leer `reglamento://aprobacion` antes de responder.


In [ ]:
respuesta = await run_llm_with_mcp_and_resources(
    "Según el reglamento de aprobación, ¿qué condiciones necesita cumplir un estudiante para aprobar la materia?"
)

print("\nRESPUESTA FINAL:")
print(respuesta)


## 14. Ejemplo 2: combinar resource + tool

Acá el modelo debería:

1. Buscar al alumno A004.
2. Calcular o interpretar su promedio.
3. Leer el reglamento.
4. Responder con criterio.


In [ ]:
respuesta = await run_llm_with_mcp_and_resources(
    "Analizá si el alumno A004 está en condiciones de aprobar según sus notas y el reglamento de aprobación."
)

print("\nRESPUESTA FINAL:")
print(respuesta)


## 15. Ejemplo 3: usar una rúbrica dinámica

El modelo debería leer `rubrica://trabajos/TP2` para explicar criterios de evaluación.


In [ ]:
respuesta = await run_llm_with_mcp_and_resources(
    "Leé la rúbrica del TP2 y explicame qué criterios se evalúan y cuál pesa más."
)

print("\nRESPUESTA FINAL:")
print(respuesta)


## 16. Ejemplo 4: listar resources disponibles

Si no sabe qué contexto existe, puede listar resources.


In [ ]:
respuesta = await run_llm_with_mcp_and_resources(
    "¿Qué documentos o recursos de contexto tenés disponibles en el servidor MCP?"
)

print("\nRESPUESTA FINAL:")
print(respuesta)
